# F3 · Madde 1 (ve Madde 5) — Aşama 5–9'un 10 tohumla tekrarı, tüm performans metrikleri
**Profesörün notu:** *Aşama 5–9 deneylerini 10 farklı tohumla tekrarlayıp tüm performans metriklerini raporla.*
*Risk–Coverage, FPR, recall ve selective risk sonuçlarını tüm modeller için tamamla.*

Her tohum (0–9) için, 02 defterinde **aynı tohumla eğitilen** merkezi AE kullanılır:
* **Aşama 5–6 (FL):** istemci bölmeleri tohuma göre çekilir; protokoller `cfg.fl_final_protocols` (F1 teşhisinden sonra
  sabitlenir; varsayılan: ara dönem protokolü + yakınsama/Adam korunur + FedProx); yalnız yerel eğitim; iki kalibrasyon.
* **Aşama 7 (TinyML):** merkezi AE ve FedAvg-IID modelleri → TFLite FP32/INT8; boyut, kaba RAM tahmini, gecikme
  (**Colab CPU'sunda; mikrodenetleyici ölçümü değildir**), FP32↔INT8 skor uyumu, eşik geçerliliği.
* **Aşama 8 (sağlamlık):** gürültü, eksik öznitelik ve bozuk öznitelik (plandaki üç senaryo), 3 tekrar, devralınan eşik /
  yeniden kalibrasyon; her deneyde F1, FPR, görülmemiş recall, kapsama ve seçici risk.
* **Aşama 9 (ablasyon):** A1–A6; her yapılandırma için FPR, recall, precision, F1, ANOMALY/UNCERTAIN/kaçan, kapsama,
  seçici risk (test kümesi ve π = %1); her model için risk–kapsama eğrisi.
* İstemci bazında sonuçlar (`fl_istemci`) ve FP32/INT8 model dosyaları (`artifacts/final_f3/tflite/`) da kaydedilir.

Raporlama: ortalama, standart sapma, tohumlar üzerinden bootstrap %95 GA, eşleştirilmiş Wilcoxon + Holm.
Güven aralıkları yalnız **eğitim tohumu** değişkenliğini yansıtır (veri bölmesi sabittir).

In [ ]:
# ---- Kurulum (her defterde aynı): Drive'ı bağla, ortak kodu içe aktar ----
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR = os.environ.get("TRUSTIDS_PROJECT", "/content/drive/MyDrive/TrustFed-TinyIDS-Final")
QUICK = os.environ.get("TRUSTIDS_QUICK", "0") == "1"   # hızlı mod YALNIZCA kodu test etmek içindir
# Devam modu: Colab oturumu kapanırsa, Drive'a kaydedilmiş tohumlar/adımlar atlanır ve kalan yerden devam edilir.
# Her şeyi baştan hesaplamak için: os.environ["TRUSTIDS_RESUME"] = "0"
RESUME = os.environ.get("TRUSTIDS_RESUME", "1") == "1"
sys.path.insert(0, PROJECT_DIR)

from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from trustids.config import Config
from trustids import utils

# Veri kümesi seçimi: varsayılan TON_IoT; ikinci veri kümesi için TRUSTIDS_CONFIG=edge_iiotset.json
CONFIG_NAME = os.environ.get("TRUSTIDS_CONFIG", "default.json")
CFG_PATH = Path(PROJECT_DIR) / "configs" / CONFIG_NAME
if CONFIG_NAME != "default.json" and not CFG_PATH.exists():
    raise FileNotFoundError(f"{CFG_PATH} bulunamadı: configs/ klasörünü de Drive'a yükleyin.")
cfg = Config.load(CFG_PATH) if CFG_PATH.exists() else Config(project_dir=PROJECT_DIR)
cfg.project_dir = PROJECT_DIR
if QUICK:
    cfg.train_seeds, cfg.validity_resplits, cfg.epochs = cfg.train_seeds[:2], 30, 15
    cfg.random_control_repeats, cfg.bootstrap_repeats = 10, 200
    cfg.lofo_seeds, cfg.fl_orig_rounds = cfg.lofo_seeds[:2], 3
    cfg.resolution_cal_windows, cfg.resolution_eval_windows = 600, 600
cfg.check()
ART = cfg.artifacts
FIG = Path(PROJECT_DIR) / cfg.figures_dir; FIG.mkdir(exist_ok=True)
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("Proje klasörü:", PROJECT_DIR, "| hızlı mod:", QUICK, "| devam modu:", RESUME)
print("Drive'da kayıtlı ilerleme:", utils.progress(cfg))

In [ ]:
import tensorflow as tf
from trustids import data as D, models as M, conformal as C, evaluation as E, fl as FL, tiny as TY, final as FZ, pipeline as PL
print(TY.self_test())                       # TFLite bu ortamda çalışıyor mu? (saatler süren eğitimden ÖNCE kontrol)
P = D.load_prepared(ART / "prepared"); X = P["X"]; FNAMES = P["preprocessor"].feature_names_
CHOSEN = utils.load_json(ART / "ae_selection.json")["chosen"]; spec = cfg.ae_variants[CHOSEN]
TAG = "F3"
CACHE = ART / "final_f3"; CACHE.mkdir(exist_ok=True)
FP = utils.fingerprint(cfg, ["fl_clients", "fl_scenarios", "fl_orig_rounds", "fl_local_epochs", "fl_fedprox_mu",
                             "fl_final_protocols", "alpha", "beta"])
RUN = utils.new_run_dir(Path(PROJECT_DIR) / cfg.runs_dir, "final_F3_asama5_9")
utils.save_json(utils.environment_info(), RUN / "environment.json"); cfg.save(RUN / "config_used.json")
d_in = X["benign_train"].shape[1]
def build(): return M.build_autoencoder(d_in, spec["hidden"], spec["dropout"], cfg.learning_rate)
CAT = {sp: FL.service_categories(X[sp], FNAMES) for sp in ["benign_train", "benign_val", "benign_cal", "benign_test"]}
COLL = P["collision"]["heldout_test"]
SEEDS = [s for s in cfg.train_seeds if (ART / "models" / f"ae_seed{s}.keras").exists() and E.scores_path(cfg, s).exists()]
missing = sorted(set(cfg.train_seeds) - set(SEEDS))
if missing: print("UYARI: bu tohumların merkezi modeli yok, önce 02 defterini tamamlayın:", missing)
def seed_inputs(seed):
    S = E.load_scores(cfg, seed)
    pick = lambda m: {sp: S[(m, sp)] for sp in PL.EV}
    return dict(central=tf.keras.models.load_model(ART / "models" / f"ae_seed{seed}.keras", compile=False),
                base=pick("ae"), iforest=pick("iforest"), latent=pick("latent"))
print("öznitelik:", d_in, "| tohumlar:", SEEDS, "| protokoller:", cfg.fl_final_protocols, "| parmak izi:", FP, "| sonuç:", RUN)

## Çalıştırma (her FL modeli ve her tohum Drive'a kaydedilir; oturum kapanırsa kaldığı yerden devam eder)

In [ ]:
for seed in SEEDS:
    f = CACHE / f"seed{seed}_{FP}.json"
    if RESUME and f.exists():
        print("tohum", seed, "Drive'da kayıtlı, atlanıyor (devam modu)"); continue
    print("tohum", seed, "çalışıyor...", flush=True)
    res = PL.run_seed(seed, cfg=cfg, X=X, cats=CAT, build=build, cache_dir=CACHE, fp=FP, resume=RESUME, coll_ho=COLL,
                      **seed_inputs(seed))
    utils.save_json(res, f)
    print("tohum", seed, "tamam (Drive'a kaydedildi)", flush=True)

## Toplama: tüm tohumlar

In [ ]:
T, DONE = PL.collect(CACHE, SEEDS, FP)
for k, df in T.items(): df.to_csv(RUN / f"{TAG}_{k}_tum_tohumlar.csv", index=False)
print("tamamlanan tohumlar:", DONE)

### Aşama 5–6 · FL (tohumlar üzerinden ortalama; testler eşleştirilmiş Wilcoxon + Holm)
`istemci_FPR_en_kötü` çok küçük bir istemciden gelebilir; bu yüzden istemci boyutuyla ağırlıklı oran da verilir. `atlanan_istemci`: verisi olmadığı için değerlendirilemeyen istemci sayısı (non-IID bölmelerde olabilir).

In [ ]:
flc = ["istemci_FPR_ort", "istemci_FPR_ağırlıklı", "istemci_FPR_en_kötü", "görülmemiş_ANOMALY", "görülmemiş_UNCERTAIN", "görülmemiş_kaçan",
       "bilinen_ANOMALY", "precision_görülmemiş", "recall_görülmemiş", "f1_görülmemiş", "coverage", "selective_risk", "atlanan_istemci"]
FLS = FZ.seed_summary(T["fl"], ["model", "senaryo", "kalibrasyon"], flc, cfg.bootstrap_repeats); FLS.to_csv(RUN / f"{TAG}_fl_ozet.csv", index=False)
EGS = T["egitim"].groupby(["senaryo", "protokol"])[["rounds_run", "epochs_equivalent", "local_steps_mean", "comm_MB", "val_loss_best"]].agg(["mean", "std"]).round(3)
EGS.to_csv(RUN / f"{TAG}_fl_egitim_ozet.csv"); display(EGS)
T["fl"]["konfig"] = T["fl"]["model"] + " | " + T["fl"]["senaryo"] + " | " + T["fl"]["kalibrasyon"]
pairs = [(f"Merkezi (yakınsama) | {s} | havuz skor", f"FedAvg · {p} | {s} | havuz skor") for s in cfg.fl_scenarios for p in cfg.fl_final_protocols]
FLT = pd.concat([FZ.paired_tests(T["fl"], "konfig", pairs, mt) for mt in ["görülmemiş_ANOMALY", "bilinen_ANOMALY", "selective_risk"]])
FLT.to_csv(RUN / f"{TAG}_fl_testler.csv", index=False)
FLS[FLS["metrik"].isin(["görülmemiş_ANOMALY", "bilinen_ANOMALY", "istemci_FPR_ağırlıklı", "istemci_FPR_en_kötü", "selective_risk"])].pivot_table(
    index=["model", "senaryo", "kalibrasyon"], columns="metrik", values="ortalama").round(4)

In [ ]:
FLT.round(4)

### Aşama 7 · TinyML (gecikme Colab CPU'sunda; mikrodenetleyici ölçümü değildir)

In [ ]:
TYS = FZ.seed_summary(T["tiny"], ["model"], ["parametre", "tflite_fp32_KB", "tflite_int8_KB", "tahmini_RAM_KB_int8",
                                             "gecikme_fp32_µs_colab", "gecikme_int8_µs_colab", "Spearman_FP32_INT8"], cfg.bootstrap_repeats)
I8S = FZ.seed_summary(T["int8"], ["model", "kural"], ["normal_FPR", "fark_α", "normal_UNCERTAIN", "görülmemiş_ANOMALY", "görülmemiş_UNCERTAIN"], cfg.bootstrap_repeats)
TYS.to_csv(RUN / f"{TAG}_tinyml_ozet.csv", index=False); I8S.to_csv(RUN / f"{TAG}_int8_gecerlilik_ozet.csv", index=False)
display(TYS.pivot_table(index="model", columns="metrik", values="ortalama").round(3))
I8S.pivot_table(index=["model", "kural"], columns="metrik", values="ortalama").round(4)

### Aşama 8 · Sağlamlık (merkezi AE): gürültü, eksik öznitelik, bozuk öznitelik
`düzey`: gürültüde standart sapma, eksik öznitelikte eksik payı, bozuk öznitelikte bozulan öznitelik sayısı (seçilen özniteliklerin değerleri örnekler arasında karıştırılır). Her satır 3 tekrarın, sonra tohumların ortalamasıdır.

In [ ]:
rob = T["rob"].groupby(["tohum", "bozulma", "düzey", "politika"], as_index=False).mean(numeric_only=True)   # önce 3 tekrarın ortalaması
RBS = FZ.seed_summary(rob, ["bozulma", "düzey", "politika"], ["normal_FPR", "normal_UNCERTAIN", "görülmemiş_ANOMALY",
                      "görülmemiş_ANOMALY+UNCERTAIN", "bilinen_ANOMALY", "f1_görülmemiş", "coverage", "selective_risk", "selective_risk_π=%1"], cfg.bootstrap_repeats)
RBS.to_csv(RUN / f"{TAG}_saglamlik_ozet.csv", index=False)
RBS.pivot_table(index=["bozulma", "düzey", "politika"], columns="metrik", values="ortalama").round(4)

### Aşama 9 · Ablasyon A1–A6: tüm metrikler (ilk tablo ortalama, ikinci tablo standart sapma, üçüncü testler)
A3q = merkezi AE'nin INT8 sürümü; INT8 etkisini FL etkisinden ayırmak için eklendi.

In [ ]:
abc = [c for c in T["abl"].columns if c not in ("yapılandırma", "tohum")]
ABS = FZ.seed_summary(T["abl"], ["yapılandırma"], abc, cfg.bootstrap_repeats); ABS.to_csv(RUN / f"{TAG}_ablasyon_ozet.csv", index=False)
pairs = [("A3 · AE + seçici", "A2 · AE (ikili)"), ("A3 · AE + seçici", "A3q · INT8 AE + seçici (merkezi)")]
for p in cfg.fl_final_protocols:
    pairs += [("A3 · AE + seçici", f"A5 · Federe AE + seçici · {p}"),
              (f"A5 · Federe AE + seçici · {p}", f"A4 · Federe AE (ikili) · {p}"),
              (f"A5 · Federe AE + seçici · {p}", f"A6 · INT8 Federe AE + seçici · {p}")]
ABT = pd.concat([FZ.paired_tests(T["abl"], "yapılandırma", pairs, mt)
                 for mt in ["görülmemiş_ANOMALY", "bilinen_ANOMALY", "selective_risk", "selective_risk_π=%1", "normal_FPR"]])
ABT.to_csv(RUN / f"{TAG}_ablasyon_testler.csv", index=False)
show = ["normal_FPR", "normal_UNCERTAIN", "bilinen_ANOMALY", "görülmemiş_ANOMALY", "görülmemiş_UNCERTAIN", "görülmemiş_kaçan", "recall_görülmemiş",
        "precision_görülmemiş", "precision_görülmemiş_π=%1", "f1_görülmemiş", "coverage", "selective_risk", "selective_risk_π=%1"]
display(ABS.pivot_table(index="yapılandırma", columns="metrik", values="ortalama", sort=False).reindex(columns=show).round(4))
display(ABS.pivot_table(index="yapılandırma", columns="metrik", values="std", sort=False).reindex(columns=show).round(4))
ABT.round(4)

### Madde 5 · Risk–kapsama eğrileri: tüm modeller (α sabit, β taranır)
Her modelin eğrisi sabit kapsama düzeylerinde okunur ve tohumlar üzerinden ortalanır. Bir modelin ulaşamadığı kapsama düzeyi boş bırakılır. π = %1 paneli saldırının nadir olduğu senaryodur (yeniden ağırlıklandırma).

In [ ]:
RCS = T["rc"].groupby(["model", "yaygınlık", "kapsama"], sort=False)["seçici_risk"].agg(["mean", "std", "count"]).reset_index()
RCS.to_csv(RUN / f"{TAG}_risk_kapsama_ozet.csv", index=False)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.2))
for ax, prev in zip(axs, ["test kümesi", "π=%1"]):
    for name, g in RCS[RCS["yaygınlık"] == prev].groupby("model", sort=False):
        g = g.dropna(subset=["mean"])
        ax.plot(g["kapsama"], g["mean"], marker="o", ms=3, lw=1.2, label=name)
    ax.set_xlabel("kapsama (otomatik karar payı)"); ax.set_ylabel("seçici risk"); ax.set_title(f"Risk–kapsama · {prev} · {len(DONE)} tohum ortalaması")
axs[1].set_yscale("log"); axs[0].legend(fontsize=6)
fig.tight_layout(); fig.savefig(RUN / f"{TAG}_risk_kapsama.png", dpi=150)
RCS[RCS["yaygınlık"] == "test kümesi"].pivot_table(index="model", columns="kapsama", values="mean", sort=False).round(4)

In [ ]:
m_ = ABS[ABS["metrik"].isin(["görülmemiş_ANOMALY", "görülmemiş_UNCERTAIN"])].pivot_table(index="yapılandırma", columns="metrik", values="ortalama", sort=False)
s_ = ABS[ABS["metrik"].isin(["görülmemiş_ANOMALY", "görülmemiş_UNCERTAIN"])].pivot_table(index="yapılandırma", columns="metrik", values="std", sort=False)
fig, ax = plt.subplots(figsize=(9, 4.6)); y = np.arange(len(m_))
ax.barh(y, 100 * m_["görülmemiş_ANOMALY"], xerr=100 * s_["görülmemiş_ANOMALY"], color="#e34948", label="ANOMALY")
ax.barh(y, 100 * m_["görülmemiş_UNCERTAIN"], left=100 * m_["görülmemiş_ANOMALY"], color="#eda100", label="UNCERTAIN")
ax.set_yticks(y); ax.set_yticklabels(m_.index, fontsize=7); ax.invert_yaxis(); ax.set_xlim(0, 100)
ax.set_xlabel("görülmemiş saldırılar (%) — ortalama ± std"); ax.set_title(f"Ablasyon, {len(DONE)} tohum"); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(RUN / f"{TAG}_ablasyon.png", dpi=150)
utils.save_json({"tohumlar": DONE, "protokoller": cfg.fl_final_protocols, "parmak_izi": FP, "öznitelik_sayısı": int(X["benign_train"].shape[1]),
                 "not": "Gecikme Colab CPU'sunda ölçüldü; mikrodenetleyici değeri değildir. GA yalnız eğitim tohumu değişkenliğini yansıtır."},
                RUN / f"{TAG}_ozet.json")
print("Sonuçlar:", RUN)

In [ ]:
# Colab: Drive'a yazılan tüm dosyaların kaydedildiğinden emin ol (oturumu kapatmadan önce mutlaka çalıştırın)
print("Kaydedilen ilerleme:", utils.progress(cfg))
if IN_COLAB:
    drive.flush_and_unmount()
    print("Drive kaydedildi ve ayrıldı. Sonraki defter Drive'ı yeniden bağlar.")